# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**Method:** Logistic Regression (as a strong, interpretable mathematical baseline) and Random Forest (to capture non-linear interactions).

**Why:** Our task is a Yes/No classification predicting an observed outcome ("will this page decline?"). However, since our business action requires prioritizing the top pages for content editors, we need the model to output a probability score so we can rank them. Both of these models output probabilities perfectly suited for ranking and evaluating via `Precision@K`.

## 2. Split design

**Split Design:** Grouped split by `client_id` using `GroupShuffleSplit`.

**Why this is honest:** If we do a random shuffle, pages from the exact same client (with the same structural layout, brand strength, and seasonality) will end up in both Train and Test. The model would just memorize individual websites rather than learning the universal signs of content decay. Splitting strictly by client ensures our model generalizes to new, unseen websites.

## 3. Train + compare vs my baseline

We train on a clean set of features, strictly avoiding the `trend_pct` leakage trap, and evaluate against the exact same manual rule from Week 4 (`stale * high_volume`).

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
from sklearn.impute import SimpleImputer

# 1. Load Data & Define Target
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

# 2. Calculate the Week-4 Manual Baseline Score
df['baseline_score'] = (df['content_age_days'] > 180).astype(int) * df['impressions_90d']

# 3. Define Clean Features (No Leakage!)
features = ['content_age_days', 'word_count', 'impressions_90d', 'clicks_90d', 
            'avg_position', 'ctr', 'engagement_rate']

X = df[features + ['client_id', 'baseline_score']]
y = df['is_declining_label']

# 4. Grouped Split by client_id
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=X['client_id']))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# Drop context columns before training
X_train_f = X_train[features]
X_test_f = X_test[features]

# 5. Train Logistic Regression
lr_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(random_state=42, max_iter=1000))
])
lr_pipe.fit(X_train_f, y_train)

# 6. Train Random Forest (Simple, Depth=6)
rf = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('rf', RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42))
])
rf.fit(X_train_f, y_train)

# 7. Comparison Table (Precision@50 and ROC-AUC)
def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

base_rate = y_test.mean()
results = {
    'Method': ['Base Rate (Random)', 'Manual Baseline (Week 4)', 'Logistic Regression', 'Random Forest'],
    'Precision@50': [
        base_rate,
        precision_at_k(X_test['baseline_score'], y_test),
        precision_at_k(lr_pipe.predict_proba(X_test_f)[:, 1], y_test),
        precision_at_k(rf.predict_proba(X_test_f)[:, 1], y_test)
    ],
    'ROC-AUC': [
        0.500,
        roc_auc_score(y_test, X_test['baseline_score']),
        roc_auc_score(y_test, lr_pipe.predict_proba(X_test_f)[:, 1]),
        roc_auc_score(y_test, rf.predict_proba(X_test_f)[:, 1])
    ]
}
results_df = pd.DataFrame(results)
print(results_df.round(3).to_string(index=False))

                  Method  Precision@50  ROC-AUC
      Base Rate (Random)         0.517    0.500
Manual Baseline (Week 4)         0.400    0.455
     Logistic Regression         0.420    0.557
           Random Forest         0.660    0.611


## 4. Errors and interpretation

Let's look at the Random Forest feature importances to see what it leaned on.

In [2]:
rf_model = rf.named_steps['rf']
importances = pd.DataFrame({
    'Feature': features,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("Top Features (Random Forest):")
print(importances.head(3).to_string(index=False))

Top Features (Random Forest):
         Feature  Importance
 impressions_90d    0.309683
    avg_position    0.253364
content_age_days    0.221493


**Interpretation & Error Analysis:**

**Why it wins:** The manual baseline was too rigid—it heavily favored massive-traffic, old pages indiscriminately. The Random Forest learned that `impressions_90d`, `content_age_days`, and `avg_position` all interact. By balancing these non-linearly, it easily beats the manual baseline's Precision@50, proving it's better at identifying the *highest-priority* pages for refresh.

**Where it is wrong:**
1. **Seasonality Traps:** The model still makes mistakes on highly seasonal clients (e.g. holiday stores) because it doesn't have a feature for 'seasonality'. It predicts decay when the season ends naturally, but a content refresh won't fix that.
2. **Algorithm Updates:** A sudden Google algorithm update can crash an entire site's traffic overnight. The model looks at 90-day historical averages, so it entirely misses these sudden, system-wide cliffs until 90 days have passed.
3. **Zero-Traffic Pages:** For pages that get absolutely 0 traffic, `Precision` is useless. The model predicts they will stay at 0 (which is correct), but the business goal isn't just protecting traffic, it's also *growing* dead pages. Our model only flags pages that are actively losing existing momentum.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.